In [0]:
dbutils.widgets.text("catalog_name",  "dbw_chinook_team", "Catalog")
dbutils.widgets.text("bronze_schema", "chinook_bronze",    "Bronze")
dbutils.widgets.text("silver_schema", "chinook_silver",    "Silver")
dbutils.widgets.text("table_name",    "all",               "Table")

In [0]:
from pyspark.sql.functions import (
    col, trim, lower, coalesce,
    lit, to_date, current_timestamp
)

CATALOG = dbutils.widgets.get("catalog_name")
BRONZE  = f"{CATALOG}.{dbutils.widgets.get('bronze_schema')}"
SILVER  = f"{CATALOG}.{dbutils.widgets.get('silver_schema')}"

print(f"✅ Bronze : {BRONZE}")
print(f"✅ Silver : {SILVER}")

In [0]:
print("=== DATA PROFILING ===")

def profile_table(table_path, table_name):
    df = spark.table(table_path)
    total = df.count()
    print(f"\n{table_name} ({total} rows):")
    for col_name in df.columns:
        null_count = df.filter(
            col(col_name).isNull()
        ).count()
        if null_count > 0:
            pct = round(null_count/total*100, 1)
            print(f"  ⚠️  {col_name}: {null_count} nulls ({pct}%)")
    dupes = total - df.dropDuplicates().count()
    if dupes > 0:
        print(f"  ⚠️  Duplicates: {dupes}")
    else:
        print(f"  ✅ No duplicates")
    return df

profile_table(f"{BRONZE}.customer",    "Customer")
profile_table(f"{BRONZE}.invoice",     "Invoice")
profile_table(f"{BRONZE}.invoiceline", "InvoiceLine")
profile_table(f"{BRONZE}.track",       "Track")
profile_table(f"{BRONZE}.employee",    "Employee")

In [0]:
print("\n=== DQX VALIDATION ===")

def validate_and_split(df, table_name, rules):
    passed = df
    all_failed = []
    
    for rule_name, condition in rules.items():
        failed = df.filter(~condition)
        failed_count = failed.count()
        if failed_count > 0:
            print(f"  ⚠️  {table_name}.{rule_name}: "
                  f"{failed_count} failed")
            all_failed.append(
                failed
                .withColumn("failed_rule", lit(rule_name))
                .withColumn("table_name",  lit(table_name))
                .withColumn("failed_at",   current_timestamp())
            )
        passed = passed.filter(condition)
    
    passed_count = passed.count()
    print(f"  ✅ {table_name}: {passed_count} passed")
    return passed, all_failed

# Customer rules
customer_df = spark.table(f"{BRONZE}.customer")
customer_passed, customer_failed = validate_and_split(
    customer_df, "Customer", {
        "CustomerId_not_null": col("CustomerId").isNotNull(),
        "Email_not_null":      col("Email").isNotNull(),
        "FirstName_not_null":  col("FirstName").isNotNull()
    }
)

# Invoice rules
invoice_df = spark.table(f"{BRONZE}.invoice")
invoice_passed, invoice_failed = validate_and_split(
    invoice_df, "Invoice", {
        "InvoiceId_not_null":  col("InvoiceId").isNotNull(),
        "CustomerId_not_null": col("CustomerId").isNotNull(),
        "Total_positive":      col("Total") > 0,
        "Date_not_null":       col("InvoiceDate").isNotNull()
    }
)

# InvoiceLine rules
invline_df = spark.table(f"{BRONZE}.invoiceline")
invline_passed, invline_failed = validate_and_split(
    invline_df, "InvoiceLine", {
        "Quantity_positive":  col("Quantity") > 0,
        "UnitPrice_positive": col("UnitPrice") > 0,
        "TrackId_not_null":   col("TrackId").isNotNull()
    }
)

# Store quarantine records
from functools import reduce
from pyspark.sql import DataFrame

all_failed_dfs = (customer_failed + 
                  invoice_failed + 
                  invline_failed)

if all_failed_dfs:
    quarantine = reduce(DataFrame.union, all_failed_dfs)
    quarantine.write.format("delta") \
        .mode("append") \
        .option("overwriteSchema","true") \
        .saveAsTable(f"{SILVER}.quarantine")
    print(f"\n⚠️  Quarantined: {quarantine.count()} records")
else:
    print("\n✅ No records quarantined — all data clean!")

In [0]:
print("\n=== WRITING SILVER TABLES ===")

# Customer
customer_silver = customer_passed \
    .withColumn("FirstName", trim(col("FirstName"))) \
    .withColumn("LastName",  trim(col("LastName"))) \
    .withColumn("Email",     lower(trim(col("Email")))) \
    .withColumn("Company",   
        coalesce(col("Company"),    lit("N/A"))) \
    .withColumn("City",      
        coalesce(col("City"),       lit("Unknown"))) \
    .withColumn("State",     
        coalesce(col("State"),      lit("N/A"))) \
    .withColumn("Country",   
        coalesce(col("Country"),    lit("Unknown"))) \
    .withColumn("PostalCode",
        coalesce(col("PostalCode"), lit("N/A")))

customer_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema","true") \
    .saveAsTable(f"{SILVER}.customer")
print(f"✅ Silver Customer: {customer_silver.count()} rows")

# Invoice
invoice_silver = invoice_passed \
    .withColumn("InvoiceDate",
        to_date(col("InvoiceDate"))) \
    .withColumn("BillingCountry",
        coalesce(col("BillingCountry"), lit("Unknown"))) \
    .withColumn("BillingState",
        coalesce(col("BillingState"),   lit("N/A"))) \
    .withColumn("Total",
        col("Total").cast("decimal(10,2)"))

invoice_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema","true") \
    .saveAsTable(f"{SILVER}.invoice")
print(f"✅ Silver Invoice: {invoice_silver.count()} rows")

# InvoiceLine
invline_silver = invline_passed \
    .withColumn("UnitPrice",
        col("UnitPrice").cast("decimal(10,2)"))

invline_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema","true") \
    .saveAsTable(f"{SILVER}.invoiceline")
print(f"✅ Silver InvoiceLine: {invline_silver.count()} rows")

# Employee
employee_df = spark.table(f"{BRONZE}.employee")
employee_silver = employee_df \
    .withColumn("FirstName", trim(col("FirstName"))) \
    .withColumn("LastName",  trim(col("LastName"))) \
    .withColumn("Email",     lower(trim(col("Email")))) \
    .withColumn("City",      
        coalesce(col("City"),    lit("Unknown"))) \
    .withColumn("State",     
        coalesce(col("State"),   lit("N/A"))) \
    .withColumn("Country",   
        coalesce(col("Country"), lit("Unknown")))

employee_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema","true") \
    .saveAsTable(f"{SILVER}.employee")
print(f"✅ Silver Employee: {employee_silver.count()} rows")

# Pass-through tables
passthrough = ["artist","album","genre","mediatype",
               "track","playlist","playlisttrack"]

for tbl in passthrough:
    df = spark.table(f"{BRONZE}.{tbl}")
    df.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema","true") \
        .saveAsTable(f"{SILVER}.{tbl}")
    print(f"✅ Silver {tbl}: {df.count()} rows")

print("\n=== SILVER LOAD COMPLETE ===")